# 16 — DBSCAN e Avaliação de Agrupamentos

> **Status:** notebook revisado e validado para estudo em JupyterLab.
>
> **Continuidade:** exemplos com dados sintéticos ou datasets integrados ao scikit-learn, sem acesso à internet.

**Objetivo geral:** construir compreensão conceitual e prática para aplicar este tema em problemas de Machine Learning, com avaliação adequada e interpretação dos resultados.


### Materiais complementares

- [Guia Técnico](./reference_materials/16_dbscan_avaliacao.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_dbscan_avaliacao.pdf)
- [Documentação da área](./README.pt-BR.md)


## 1. Fundamentos, objetivos e contexto

**Objetivos.** Entender grupos por densidade, parâmetros `eps` e `min_samples`, pontos núcleo/borda/ruído, e métricas internas/externas.

DBSCAN identifica regiões densas e pode apontar ruído com rótulo -1. É útil para formas não convexas, mas sensível à escala e à escolha de `eps`; densidades muito diferentes em um mesmo dataset são um desafio. Métricas externas como ARI usam rótulos verdadeiros quando existentes; silhouette não é definida para menos de dois clusters válidos.


## 2. Fundamentos matemáticos

Definição intuitiva: um ponto núcleo tem pelo menos `min_samples` observações na sua vizinhança $N_\varepsilon(x)=\{z:d(x,z)\le\varepsilon\}$ (incluindo o próprio ponto em implementações usuais). Silhouette individual: $s(i)=(b(i)-a(i))/\max\{a(i),b(i)\}$.

**Conexão com disciplinas anteriores:** revise os conceitos matemáticos e estatísticos correspondentes. Não repetimos integralmente os capítulos de Python, visualização e inferência estatística.


## 3. Experimento guiado: preparando a pergunta

**Antes do código**, identifique: (a) quais dados são de treino, (b) qual informação poderá entrar no modelo, (c) que métrica ou propriedade será observada, e (d) que conclusão não é permitida pelos resultados. Estes exemplos são didáticos, com semente fixa onde aplicável.


In [1]:
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN,KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score,silhouette_score
import numpy as np
X,y=make_moons(n_samples=300,noise=.06,random_state=42)
X=StandardScaler().fit_transform(X)
for eps in [.15,.25,.4]:
    labels=DBSCAN(eps=eps,min_samples=5).fit_predict(X)
    n_clusters=len(set(labels)-{-1});noise=(labels==-1).sum()
    ari=adjusted_rand_score(y,labels)
    print('eps',eps,'clusters',n_clusters,'ruído',noise,'ARI (comparação didática)',round(ari,3))

eps 0.15 clusters 16 ruído 26 ARI (comparação didática) 0.299
eps 0.25 clusters 2 ruído 0 ARI (comparação didática) 1.0
eps 0.4 clusters 2 ruído 0 ARI (comparação didática) 1.0


### Como interpretar

Os números são resultados específicos do experimento e da amostra, não garantias universais. Verifique o que mudou, qual método foi usado e qual é o cenário de aplicação. Compare com a teoria matemática e não confunda desempenho de treino, validação e teste.


## 4. Segundo experimento ou extensão controlada

Mude apenas uma condição principal (algoritmo, parâmetro ou tipo de dado), quando possível, e registre o efeito. Não ajuste decisões com base no teste final.


In [2]:
from sklearn.cluster import KMeans
lab=DBSCAN(eps=.25,min_samples=5).fit_predict(X)
km=KMeans(n_clusters=2,n_init=10,random_state=42).fit_predict(X)
print('ARI DBSCAN',round(adjusted_rand_score(y,lab),3),'ARI KMeans',round(adjusted_rand_score(y,km),3))
valid=lab!=-1
if len(set(lab[valid]))>=2:
    print('Silhouette nos pontos não ruído:',round(silhouette_score(X[valid],lab[valid]),3))

ARI DBSCAN 1.0 ARI KMeans 0.479
Silhouette nos pontos não ruído: 0.384


## 5. Exercício resolvido

**Enunciado:** reforce o conceito central com um exemplo menor, cujo resultado pode ser inspecionado diretamente.


In [3]:
# Exercício resolvido: contar pontos de ruído
labels=np.array([0,0,-1,1,1,-1,1]);print('ruído:',int(np.sum(labels==-1)),'clusters:',len(set(labels)-{-1}))

ruído: 2 clusters: 2


## 6. Interpretação, limitações e erros comuns

Métricas externas requerem rótulos de referência e só fazem sentido em experimentos onde eles existem. Silhouette pode ter interpretação ruim para estruturas não convexas como luas. DBSCAN não atribui automaticamente novos pontos por `predict` no estimador padrão.

**Perguntas de revisão:**

1. Qual hipótese implícita sustenta o procedimento usado?
2. Que erro de implementação ou de validação poderia tornar o resultado artificialmente otimista?
3. Como você mudaria o experimento para refletir dados reais?


## 7. Consulta rápida

- `eps`: raio de vizinhança; `min_samples`: densidade mínima.
- Ruído recebe -1.
- Escalonamento e distância mudam a definição de densidade.
- Avaliação interna não é sinônimo de validação científica dos grupos.


## 8. Exercícios adicionais (para fazer no JupyterLab)

**Exercício A — sensibilidade:** altere um hiperparâmetro/hipótese e apresente uma tabela pequena com resultados e explicação. **Exercício B — revisão crítica:** construa um exemplo de uso incorreto (sem aplicá-lo a dados reais) e explique o procedimento correto. **Exercício C — comunicação:** escreva um parágrafo sobre escopo, métricas, suposições e limitações do método.

**Critério de aprovação:** códigos executam do começo ao fim, não há seleção pelo teste final e cada resultado possui interpretação.


## 9. Fontes e continuidade curricular

**Origem principal das aulas:** 01-introducao.pdf (menção DBSCAN); 15-agrupamento.pdf (base de clustering).

**Referências de atualização:** documentação de `scikit-learn` (<https://scikit-learn.org/stable/>); explicações matemáticas redigidas para esta reconstrução. Outros pacotes (Optuna, XGBoost, SHAP, imbalanced-learn) não são necessários para executar este capítulo.



## Aprofundamento v2 — auditoria e aplicação

**Ponto de atenção deste capítulo:** ruído e densidade. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **ruído e densidade** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **ruído e densidade** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.


In [4]:
# Evidência gráfica: ruído e agrupamento do DBSCAN, sem dados externos.
from pathlib import Path
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
X_vis, _ = make_moons(n_samples=240, noise=0.09, random_state=42)
X_vis = StandardScaler().fit_transform(X_vis)
classes_vis = DBSCAN(eps=0.3, min_samples=5).fit_predict(X_vis)
Path("outputs").mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(X_vis[:, 0], X_vis[:, 1], c=classes_vis, s=18)
ax.set(title="DBSCAN — dados sintéticos e pontos de ruído", xlabel="Atributo 1 padronizado", ylabel="Atributo 2 padronizado")
fig.tight_layout()
fig.savefig("outputs/16_dbscan_sintetico.png", dpi=160)
plt.close(fig)
print("Gerado: outputs/16_dbscan_sintetico.png")

Gerado: outputs/16_dbscan_sintetico.png
